# Multi-Model Embedding + Gemma 4 12B Chat Server (Ollama)
GPU 0 loads the top 5 general-purpose English embedding models (by MTEB
score) and serves them over FastAPI (`/models` to list, `/embed` to run
inference) -- around 6GB used, ~10GB free there. GPU 1 is fully free
(16GB) and runs the chat model through Ollama.

Embedding models: Qwen3-Embedding-0.6B, EmbeddingGemma-300M,
gte-large-en-v1.5, mxbai-embed-large-v1, bge-large-en-v1.5 (GPU 0).
Chat model: `gemma4:12b` -- Google's official Gemma 4 12B, pulled from
Ollama's own model library (~8GB, fits GPU 1 on its own). Using an
official library tag rather than a third-party GGUF repack, since the
previous attempt (a community Qwen3.8-27B repack) silently failed to
load -- an unverified community tag can simply not exist or be broken;
`gemma4:12b` is a known-good official tag.

**Checking whether things are actually up:** `/health` is a fast check
(what's loaded/pulled, no inference run). To actually *validate* each
model is working, not just present, use:
- Embeddings: `GET {public_url}/embed/status` -- runs a real test embed
  against every loaded model
- LLM: `GET {public_url}/chat/status` -- runs a real test generation
  against the chat model

(`{public_url}` is printed by Cell 5 once it's up -- it's your reserved
ngrok URL every run when the command carries `--url`, or a random one if
it fell back to Cloudflare.)

**One-time setup before running:**
1. EmbeddingGemma is gated on Hugging Face -- accept the license at
   https://huggingface.co/google/embeddinggemma-300m and create a read
   token at https://huggingface.co/settings/tokens
2. Sign up free at https://ngrok.com and copy your authtoken from
   https://dashboard.ngrok.com/get-started/your-authtoken
3. Create the endpoint in the ngrok dashboard (Endpoints -> New -> Agent
   Endpoint). The dashboard hands you the exact `ngrok` command to run,
   e.g. `ngrok http 8080 --url=https://your-name.ngrok-free.app`, plus a
   traffic policy if you wrote one. The notebook runs that command as
   given and changes only the local port (Cell 5 picks a free one).
4. In this notebook: Add-ons -> Secrets -> add:
   - `HF_TOKEN` (required for EmbeddingGemma)
   - `NGROK_AUTHTOKEN` (required for ngrok; never printed)
   - `NGROK_COMMAND` -- the command the dashboard handed you, verbatim
   - `NGROK_TRAFFIC_POLICY` (optional) -- the policy YAML, if the command
     references `policy.yaml`

   Without `NGROK_COMMAND` the notebook runs `ngrok http <port>`, adding
   `--url=` from an `NGROK_URL` (or the older `NGROK_DOMAIN`) secret when
   one is set. With no URL at all, ngrok assigns one and Cell 5 reports it.

Run all 6 cells top to bottom. Cell 4 installs Ollama, starts it as a
background server, and pulls the ~8GB model -- this is the slow step
(download time). Cell 5 picks a free local port (fixes "address already in use" from a
leftover server in the same kernel), checks the local server answers
*before* starting ngrok (otherwise the endpoint comes up looking healthy
and only fails on the first public request), starts the ngrok agent in the
background with your dashboard command, and then requests the public URL
to prove it serves the same response as the local server -- not an ngrok
error page. If ngrok cannot start (no authtoken, URL already online
elsewhere, bad policy), it says why and falls back to a Cloudflare Quick
Tunnel (random URL, no account needed). It prints an
`EXECUTION COMPLETE -- SERVER IS LIVE AND READY` banner once the public
URL has actually answered, and which tunnel is active (ngrok = your URL,
Cloudflare = random).

The ngrok endpoint lives only while the agent process runs: stopping the
notebook takes the URL down. For an always-on endpoint with no local
process, use an ngrok cloud endpoint instead (https://ngrok.com/docs/start).

This notebook keeps running (and burning your weekly GPU quota) until it
hits MAX_RUNTIME_HOURS or you stop the session manually from
kaggle.com/code -> your kernel -> Stop Session (there is no API call to
stop a session remotely, so build that into your local workflow). When it
stops, Cell 6 prints a `SESSION STOPPED` message.

In [ ]:
# Cell 1 -- install dependencies and the ngrok agent
# Recent sentence-transformers/transformers are required for Qwen3-Embedding
# and EmbeddingGemma (Gemma3 backbone) support. The ollama python client
# talks to the local Ollama server started in Cell 4.
#
# The tunnel uses the ngrok agent itself (the `ngrok` CLI), not a Python
# wrapper, so the command the ngrok dashboard hands you runs exactly as given.
!pip install -q -U "sentence-transformers>=3.4.0" "transformers>=4.51.0" \
    huggingface_hub fastapi "uvicorn[standard]" einops ollama requests
!curl -fsSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz | tar -xz -C /usr/local/bin
!ngrok version
print("dependencies installed")

In [ ]:
# Cell 2 -- load secrets and authenticate (Hugging Face + ngrok)
# Secrets (Add-ons -> Secrets): HF_TOKEN, NGROK_AUTHTOKEN, NGROK_COMMAND,
# optional NGROK_TRAFFIC_POLICY / NGROK_URL. See the setup notes above.
# The ngrok authtoken is handed straight to `ngrok config add-authtoken` and
# is never printed or written anywhere else.
import subprocess

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()

def optional_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        return None
    return value.strip() if value and value.strip() else None

hf_token = optional_secret("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Hugging Face: authenticated")
else:
    print("WARNING: no HF_TOKEN secret found -- EmbeddingGemma will fail to load")

# The exact command the ngrok dashboard generated for your Agent Endpoint.
ngrok_command = optional_secret("NGROK_COMMAND")
ngrok_policy = optional_secret("NGROK_TRAFFIC_POLICY")
# Only used when there is no NGROK_COMMAND. NGROK_DOMAIN is the older name.
ngrok_url = optional_secret("NGROK_URL") or optional_secret("NGROK_DOMAIN")
if ngrok_url and not ngrok_url.startswith(("http://", "https://")):
    ngrok_url = "https://" + ngrok_url

ngrok_ready = False
authtoken = optional_secret("NGROK_AUTHTOKEN")
if authtoken:
    result = subprocess.run(
        ["ngrok", "config", "add-authtoken", authtoken],
        capture_output=True, text=True,
    )
    del authtoken
    ngrok_ready = result.returncode == 0
    print("ngrok: authtoken saved" if ngrok_ready else "ngrok: could not save the authtoken")
else:
    print("ngrok: no NGROK_AUTHTOKEN secret -- Cell 5 will use the Cloudflare fallback")

if ngrok_command:
    print(f"ngrok: will run the dashboard command: {ngrok_command}")
elif ngrok_url:
    print(f"ngrok: no NGROK_COMMAND; will run `ngrok http <port> --url={ngrok_url}`")
else:
    print("ngrok: no NGROK_COMMAND or NGROK_URL; ngrok will assign a URL")

In [ ]:
# Cell 3 -- model registry + load all 5 embedding models onto GPU 0
# Top 5 general-purpose English embedding models by MTEB score. All are
# sentence-transformers compatible and all 5 comfortably fit on a single
# 16GB GPU (largest is ~600M params), so they're all pinned to GPU 0 and
# all load at startup. GPU 1 is left free for the Ollama chat model (Cell 4).
#
# query_prefix / passage_prefix: these models were trained with instruction
# prefixes for asymmetric retrieval (query vs. passage). /embed lets the
# caller pick which one to apply, or none for symmetric use (e.g. clustering,
# STS).
import torch
from sentence_transformers import SentenceTransformer

MODEL_REGISTRY = {
    "qwen3-embedding-0.6b": {
        "repo": "Qwen/Qwen3-Embedding-0.6B",
        "query_prefix": (
            "Instruct: Given a web search query, retrieve relevant passages "
            "that answer the query\nQuery: "
        ),
        "passage_prefix": "",
        "trust_remote_code": False,
    },
    "embeddinggemma-300m": {
        "repo": "google/embeddinggemma-300m",
        "query_prefix": "task: search result | query: ",
        "passage_prefix": "title: none | text: ",
        "trust_remote_code": False,
    },
    "gte-large-en-v1.5": {
        "repo": "Alibaba-NLP/gte-large-en-v1.5",
        "query_prefix": "",
        "passage_prefix": "",
        "trust_remote_code": True,
    },
    "mxbai-embed-large-v1": {
        "repo": "mixedbread-ai/mxbai-embed-large-v1",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "passage_prefix": "",
        "trust_remote_code": False,
    },
    "bge-large-en-v1.5": {
        "repo": "BAAI/bge-large-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "passage_prefix": "",
        "trust_remote_code": False,
    },
}

n_gpus = torch.cuda.device_count()
EMBEDDING_DEVICE = "cuda:0" if n_gpus > 0 else "cpu"
print(f"visible GPUs: {n_gpus} -- all embedding models pinned to {EMBEDDING_DEVICE}\n")

LOADED = {}
for name, cfg in MODEL_REGISTRY.items():
    device = EMBEDDING_DEVICE
    print(f"loading {name} ({cfg['repo']}) on {device} ...")
    model = SentenceTransformer(
        cfg["repo"],
        device=device,
        trust_remote_code=cfg["trust_remote_code"],
        token=hf_token,
    )
    LOADED[name] = {
        "model": model,
        "device": device,
        "dim": model.get_sentence_embedding_dimension(),
        "max_seq_length": model.max_seq_length,
        "repo": cfg["repo"],
        "query_prefix": cfg["query_prefix"],
        "passage_prefix": cfg["passage_prefix"],
    }
    print(f"  -> ready, dim={LOADED[name]['dim']}, max_seq_length={LOADED[name]['max_seq_length']}")

print(f"\nall {len(LOADED)} models loaded: {list(LOADED.keys())}")

In [ ]:
# Cell 4 -- install Ollama, start it, pull Gemma 4 12B
# gemma4:12b is an OFFICIAL tag in Ollama's own model library (not a
# third-party repack), ~8GB -- this is also why the previous model
# silently failed: that community GGUF tag wasn't guaranteed to exist.
# An official library tag removes that failure mode. At ~8GB it fits
# GPU 1 on its own with plenty of headroom (no cross-GPU split needed),
# while Ollama still auto-places it based on actual free VRAM.
import subprocess
import time

import ollama

print("installing Ollama ...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

print("starting Ollama server ...")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

ollama_client = ollama.Client(host="http://127.0.0.1:11434")
for _ in range(30):
    try:
        ollama_client.list()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not come up in time")
print("Ollama server ready")

CHAT_MODEL = "gemma4:12b"  # official Ollama library tag, ~8GB
print(f"pulling {CHAT_MODEL} ...")
ollama_client.pull(CHAT_MODEL)

# Pulling only means the weights are on disk -- confirm it actually runs
# by sending one real, cheap generation before calling it ready.
print("running a live test generation to confirm the model actually loads and infers ...")
test = ollama_client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Reply with just the word: ready"}],
    options={"num_predict": 5},
)
print(f"chat model ready: {CHAT_MODEL}  (test reply: {test['message']['content']!r})")


In [ ]:
# Cell 5 -- define the API, start the server, open the tunnel
# Follows ngrok's Agent Endpoint setup: confirm the local app responds, run
# the dashboard's command in the background (only the local port changed),
# then prove the PUBLIC URL serves the same response. If ngrok cannot come
# up, falls back to a Cloudflare Quick Tunnel (no account needed, but the
# URL is random and changes every run).
import os
import json
import re
import shlex
import socket
import stat
import subprocess
import threading
import time
import urllib.request

import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Kaggle Embedding + Chat Server")

class EmbedRequest(BaseModel):
    model: str
    texts: list[str]
    prefix: str = "none"   # "query" | "passage" | "none"
    normalize: bool = True

class ChatRequest(BaseModel):
    messages: list[dict]
    temperature: float = 0.7
    max_tokens: int = 1024

@app.get("/health")
def health():
    # Fast liveness check: lists what's loaded/pulled, no inference performed.
    # For a REAL functional check (actually runs a test call), use
    # /embed/status and /chat/status below.
    try:
        ollama_models = [m["model"] for m in ollama_client.list().get("models", [])]
        chat_ready = CHAT_MODEL in ollama_models
    except Exception:
        chat_ready = False
    return {
        "status": "ok",
        "embedding_models_loaded": list(LOADED.keys()),
        "chat_model_pulled": chat_ready,
    }

@app.get("/embed/status")
def embed_status():
    # Runs a real test embedding (not just "is it loaded") against every
    # model, so a model that's present but broken shows up as broken.
    results = {}
    for name, info in LOADED.items():
        try:
            start = time.time()
            vec = info["model"].encode(["status check"], convert_to_numpy=True)
            results[name] = {
                "up": True,
                "dim": len(vec[0]),
                "latency_ms": round((time.time() - start) * 1000, 1),
            }
        except Exception as e:
            results[name] = {"up": False, "error": str(e)}
    return {"all_up": all(r["up"] for r in results.values()), "models": results}

@app.get("/chat/status")
def chat_status():
    # Runs a real, cheap test generation against the LLM, not just a
    # "has it been pulled" check -- catches a model that's downloaded but
    # failing to actually load/run (OOM, corrupt pull, etc).
    try:
        start = time.time()
        response = ollama_client.chat(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": "Reply with just the word: ready"}],
            options={"num_predict": 5},
        )
        return {
            "up": True,
            "model": CHAT_MODEL,
            "latency_ms": round((time.time() - start) * 1000, 1),
            "sample_reply": response["message"]["content"],
        }
    except Exception as e:
        return {"up": False, "model": CHAT_MODEL, "error": str(e)}

@app.get("/models")
def list_models():
    return {
        name: {
            "repo": info["repo"],
            "dim": info["dim"],
            "max_seq_length": info["max_seq_length"],
            "device": info["device"],
            "supports_prefix": bool(info["query_prefix"] or info["passage_prefix"]),
        }
        for name, info in LOADED.items()
    }

@app.post("/embed")
def embed(req: EmbedRequest):
    if req.model not in LOADED:
        raise HTTPException(
            status_code=404,
            detail=f"unknown model '{req.model}'. Available: {list(LOADED.keys())}",
        )
    if not req.texts:
        raise HTTPException(status_code=400, detail="texts must be a non-empty list")

    info = LOADED[req.model]
    prefix = ""
    if req.prefix == "query":
        prefix = info["query_prefix"]
    elif req.prefix == "passage":
        prefix = info["passage_prefix"]

    inputs = [prefix + t for t in req.texts]
    vectors = info["model"].encode(
        inputs,
        normalize_embeddings=req.normalize,
        convert_to_numpy=True,
    )
    return {
        "model": req.model,
        "dim": info["dim"],
        "embeddings": vectors.tolist(),
    }

@app.post("/chat")
def chat(req: ChatRequest):
    try:
        response = ollama_client.chat(
            model=CHAT_MODEL,
            messages=req.messages,
            options={"temperature": req.temperature, "num_predict": req.max_tokens},
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"chat inference error: {e}")

    return {
        "model": CHAT_MODEL,
        "content": response["message"]["content"],
        "usage": {
            "prompt_tokens": response.get("prompt_eval_count"),
            "completion_tokens": response.get("eval_count"),
        },
    }

# -- find a free port (fixes "address already in use" from a server left
#    over by a previous run in the same kernel) instead of hardcoding 8000 --
def find_free_port(start=8000, tries=10):
    for port in range(start, start + tries):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                s.bind(("0.0.0.0", port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"no free port found in range {start}-{start + tries}")

SERVER_PORT = find_free_port()
print(f"using local port {SERVER_PORT}")

# -- start the API server in a background thread --
config = uvicorn.Config(app, host="0.0.0.0", port=SERVER_PORT, log_level="warning")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(3)

# -- 1. confirm the local app responds BEFORE starting a tunnel. If nothing
#    is listening, an ngrok endpoint still comes up looking healthy and only
#    fails when someone loads the public URL. --
def get_json(url, timeout=10, headers=None):
    request = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(request, timeout=timeout) as r:
        return r.status, r.read().decode("utf-8", "replace")

local_health = None
for _ in range(20):
    try:
        status, body = get_json(f"http://127.0.0.1:{SERVER_PORT}/health", timeout=5)
        if status == 200:
            local_health = json.loads(body)
            break
    except Exception:
        time.sleep(1)
if local_health is None:
    raise RuntimeError(f"the local server is not answering on port {SERVER_PORT}; not starting a tunnel")
print(f"local server answers on port {SERVER_PORT}: {local_health['status']}")

# -- 2. the ngrok command: the dashboard's, verbatim, except the local port --
NGROK_DIR = "/kaggle/working/ngrok"     # policy.yaml lives here; ngrok runs from here
NGROK_LOG = os.path.join(NGROK_DIR, "ngrok.log")
NGROK_API = "http://127.0.0.1:4040/api/tunnels"
os.makedirs(NGROK_DIR, exist_ok=True)

def with_local_port(upstream, port):
    """Change only the port of the upstream: 8080, localhost:8080 or
    http://localhost:8080 all keep their host and scheme."""
    if upstream.isdigit():
        return str(port)
    if re.search(r":\d+/?$", upstream):
        return re.sub(r":\d+(/?)$", rf":{port}\1", upstream)
    return f"{upstream.rstrip('/')}:{port}"

def ngrok_args(port):
    if ngrok_command:
        args = shlex.split(ngrok_command)
        # The upstream is the one positional argument (port, host:port or a
        # URL), before or after the flags; a flag's own value never counts.
        upstream = [
            i for i in range(2, len(args))
            if re.fullmatch(r"\d+|[\w.-]+:\d+|https?://\S+|file://\S+", args[i])
            and not (args[i - 1].startswith("--") and "=" not in args[i - 1] and i - 1 >= 2)
        ]
        if args[:2] != ["ngrok", "http"] or len(upstream) != 1:
            raise ValueError("NGROK_COMMAND must be the `ngrok http <port> ...` command the dashboard generated")
        args[upstream[0]] = with_local_port(args[upstream[0]], port)
    else:
        args = ["ngrok", "http", str(port)]
        if ngrok_url:
            args.append(f"--url={ngrok_url}")
        if ngrok_policy:
            args.append("--traffic-policy-file=policy.yaml")
    # Agent logging only, so failures can be explained; no endpoint setting changes.
    return args + ["--log=stdout", "--log-format=json"]

def url_in(args):
    for i, arg in enumerate(args):
        if arg.startswith("--url="):
            return arg.split("=", 1)[1]
        if arg == "--url" and i + 1 < len(args):
            return args[i + 1]
    return None

def policy_summary(text):
    """Auth or limits the policy adds, so a 401 on the first request is expected, not 'broken'."""
    found = re.findall(r"type:\s*([a-z0-9-]+)", text)
    gates = [t for t in found if t in ("basic-auth", "oauth", "openid-connect", "jwt-validation", "rate-limit",
                                       "restrict-ips", "deny")]
    return sorted(set(gates))

def explain_ngrok_failure(log_text):
    if "ERR_NGROK_334" in log_text or "already online" in log_text:
        return ("this URL is already online in another ngrok session (another notebook, terminal or kernel). "
                "Stop that session rather than picking a new URL.")
    if "ERR_NGROK_4018" in log_text or "ERR_NGROK_105" in log_text or "authtoken" in log_text.lower():
        return "the authtoken was rejected -- re-copy NGROK_AUTHTOKEN from the ngrok dashboard"
    if "policy" in log_text.lower():
        return "the traffic policy failed to load -- check NGROK_TRAFFIC_POLICY (it is written to policy.yaml)"
    return "see " + NGROK_LOG

def start_ngrok(port):
    global ngrok_proc
    if not ngrok_ready:
        raise RuntimeError("ngrok has no authtoken (NGROK_AUTHTOKEN)")
    # Re-running this cell in the same kernel: stop the agent this notebook
    # started before, so its URL is free again (never pick a different URL).
    if globals().get("ngrok_proc") is not None and ngrok_proc.poll() is None:
        ngrok_proc.terminate()
        ngrok_proc.wait(timeout=10)

    args = ngrok_args(port)
    if "policy.yaml" in " ".join(args):
        if not ngrok_policy:
            raise RuntimeError("the command references policy.yaml but there is no NGROK_TRAFFIC_POLICY secret")
        with open(os.path.join(NGROK_DIR, "policy.yaml"), "w") as f:
            f.write(ngrok_policy)
        gates = policy_summary(ngrok_policy)
        if gates:
            print(f"traffic policy adds {', '.join(gates)}: the first request must satisfy it (expect 401/403 otherwise)")

    print("starting ngrok:", " ".join(args[:-2]))
    log = open(NGROK_LOG, "w")
    ngrok_proc = subprocess.Popen(args, cwd=NGROK_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)

    # With --url that is the address; without it ngrok assigns one at startup.
    expected = url_in(args)
    for _ in range(30):
        time.sleep(1)
        if ngrok_proc.poll() is not None:
            raise RuntimeError("ngrok exited: " + explain_ngrok_failure(open(NGROK_LOG).read()))
        try:
            _, body = get_json(NGROK_API, timeout=3)
            tunnels = json.loads(body).get("tunnels", [])
        except Exception:
            continue
        urls = [t["public_url"] for t in tunnels if t.get("public_url", "").startswith("https://")]
        urls = urls or [t["public_url"] for t in tunnels if t.get("public_url")]
        if urls:
            if expected and expected.rstrip("/") not in [u.rstrip("/") for u in urls]:
                print(f"note: ngrok reports {urls[0]}, the command asked for {expected}")
            return urls[0]
    ngrok_proc.terminate()
    raise RuntimeError("ngrok did not report an endpoint within 30 s: " + explain_ngrok_failure(open(NGROK_LOG).read()))

def verify_public(url):
    """Request the PUBLIC URL and confirm it is our app answering, not an ngrok error page."""
    headers = {"ngrok-skip-browser-warning": "1", "User-Agent": "kaggle-embedding-server-check"}
    try:
        status, body = get_json(f"{url}/health", timeout=20, headers=headers)
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", "replace")
        if "ERR_NGROK_8012" in body:
            return False, ("ERR_NGROK_8012: traffic reached the agent but not the app -- the local server stopped "
                           "or is on another port (re-check step 1 before changing the endpoint)")
        return False, f"HTTP {e.code} from the public URL" + (" (a traffic policy may require auth)" if e.code in (401, 403) else "")
    except Exception as e:
        return False, f"public URL did not answer: {e}"
    try:
        public = json.loads(body)
    except ValueError:
        return False, "the public URL answered with something other than this server (an error page?)"
    same = public.get("embedding_models_loaded") == local_health["embedding_models_loaded"]
    return same, (f"HTTP {status}, same /health as the local server" if same
                  else "the public URL answers, but not with this server's /health")

# -- 3. open a tunnel: ngrok first, Cloudflare Quick Tunnel as a fallback --
public_url = None
tunnel_mode = None
cf_proc = None
ngrok_proc = globals().get("ngrok_proc")
public_check = None

try:
    public_url = start_ngrok(SERVER_PORT)
    tunnel_mode = "ngrok"
    print(f"ngrok endpoint up: {public_url}")
except Exception as e:
    print(f"ngrok endpoint did not come up ({e})")
    print("falling back to a Cloudflare Quick Tunnel -- this URL is RANDOM and will change next run.")

    CLOUDFLARED_PATH = "/kaggle/working/cloudflared"
    if not os.path.exists(CLOUDFLARED_PATH):
        cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(cf_url, CLOUDFLARED_PATH)
        st = os.stat(CLOUDFLARED_PATH)
        os.chmod(CLOUDFLARED_PATH, st.st_mode | stat.S_IEXEC)

    cf_proc = subprocess.Popen(
        [CLOUDFLARED_PATH, "tunnel", "--url", f"http://localhost:{SERVER_PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    url_pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")
    for line in cf_proc.stdout:
        m = url_pattern.search(line)
        if m:
            public_url = m.group(0)
            break
    tunnel_mode = "cloudflare"
    print(f"Cloudflare Quick Tunnel up: {public_url}")

if public_url is None:
    raise RuntimeError("both ngrok and the Cloudflare fallback failed to produce a public URL")

# -- 4. prove the public URL serves this app (Cloudflare can take a few s) --
for attempt in range(6):
    ok, public_check = verify_public(public_url)
    if ok:
        break
    time.sleep(5)

print("=" * 60)
if ok:
    print("EXECUTION COMPLETE -- SERVER IS LIVE AND READY")
else:
    print("TUNNEL STARTED BUT THE PUBLIC URL CHECK FAILED -- see below")
print("=" * 60)
print(f"Tunnel         : {tunnel_mode}" + (" (your ngrok endpoint)" if tunnel_mode == "ngrok" else " (RANDOM -- changes next run)"))
print(f"Public URL     : {public_url}")
print(f"Public check   : {public_check}")
print(f"Embedding models (GPU 0, ~6GB): {list(LOADED.keys())}")
print(f"Chat model (GPU 1, Ollama, ~8GB): {CHAT_MODEL}")
print("-" * 60)
print(f"List models  : GET  {public_url}/models")
print(f"Embed        : POST {public_url}/embed")
print(f"  example    : {{\"model\": \"bge-large-en-v1.5\", \"texts\": [\"hello world\"]}}")
print(f"Chat         : POST {public_url}/chat")
print(f"  example    : {{\"messages\": [{{\"role\": \"user\", \"content\": \"hi\"}}]}}")
print("-" * 60)
print(f"Overall health      : GET {public_url}/health          (fast, no inference)")
print(f"Validate embeddings : GET {public_url}/embed/status     (runs a real test embed)")
print(f"Validate LLM        : GET {public_url}/chat/status      (runs a real test generation)")
print(f"Try it yourself     : curl -H 'ngrok-skip-browser-warning: 1' {public_url}/health")
print("-" * 60)
print("To stop: interrupt Cell 6, or stop the session (kaggle.com/code -> Stop Session).")
print("The ngrok URL goes down when its agent process stops.")
print("=" * 60)
print("Run the next cell to keep this session alive and serving.")

In [ ]:
# Cell 6 -- keep the session alive with a hard self-timeout safety net.
# There is no API to stop a Kaggle session remotely, so this timeout is
# what stops the notebook (and your GPU-hour spend) if you forget to stop
# it manually. Tune MAX_RUNTIME_HOURS to taste. Interrupting this cell also
# shuts everything down cleanly.

MAX_RUNTIME_HOURS = 8

start = time.time()
try:
    while time.time() - start < MAX_RUNTIME_HOURS * 3600:
        time.sleep(30)
        if tunnel_mode == "ngrok" and ngrok_proc.poll() is not None:
            print("ngrok agent exited: " + explain_ngrok_failure(open(NGROK_LOG).read()))
            break
        if tunnel_mode == "cloudflare" and cf_proc is not None and cf_proc.poll() is not None:
            print("cloudflare tunnel process died, exiting")
            break
finally:
    if ngrok_proc is not None and ngrok_proc.poll() is None:
        ngrok_proc.terminate()
    if cf_proc is not None and cf_proc.poll() is None:
        cf_proc.terminate()
    server.should_exit = True
    ollama_proc.terminate()
    elapsed_min = (time.time() - start) / 60
    print("=" * 60)
    print(f"SESSION STOPPED -- server and tunnel shut down after {elapsed_min:.1f} min")
    print(f"API is no longer reachable at {public_url}")
    print("=" * 60)